In [1]:
import pandas as pd

In [2]:
WASHER = pd.read_csv('lv2-2412.csv')

In [3]:
WASHER.shape

(205, 9)

In [5]:
WASHER.head()

,PRD_ID,WEIGHT,ENERGY_CONS,WATER_CONS,SPIN_SPEED,NOISE,MOTOR_TYPE,MATERIAL,SCORE
0,1,251.35,209.10,26.55,1238.97,79.68,A,A,68
1,2,250.84,208.93,29.32,1240.25,81.78,A,A,75
2,3,254.01,215.46,29.28,1351.34,68.60,A,A,72
3,4,258.55,206.91,25.93,1344.86,77.76,B,B,71
4,5,258.56,207.93,26.26,1343.22,76.79,C,B,75


# Q1

In [6]:
df_1 = WASHER.copy()

# 1-1

In [7]:
DF40_A = df_1[df_1['MOTOR_TYPE'] == 'A']

In [8]:
DF40_B = df_1[df_1['MOTOR_TYPE'] == 'B']

# 1-2

In [9]:
targets = ['ENERGY_CONS', 'WATER_CONS', 'NOISE']

In [10]:
COR_A = []
COR_B = []

In [11]:
from scipy.stats import pearsonr

In [13]:
for target in targets:
    p_a = pearsonr(x=DF40_A[target], y=DF40_A['SCORE'])
    COR_A.append(p_a[0])
    p_b = pearsonr(x=DF40_B[target], y=DF40_B['SCORE'])
    COR_B.append(p_b[0])

# 1-3

In [14]:
COR_A

[np.float64(0.6664521262737249),
 np.float64(0.5167342347211216),
 np.float64(0.3721840631868748)]

In [15]:
COR_B

[np.float64(0.6602443967298892),
 np.float64(0.17874088955641126),
 np.float64(0.20501187669651982)]

In [16]:
GAP = []

for i in range(3):
    GAP.append(abs(COR_A[i] - COR_B[i]))

In [17]:
GAP

[np.float64(0.006207729543835638),
 np.float64(0.3379933451647103),
 np.float64(0.16717218649035498)]

In [19]:
SUM = 0

for i in range(3):
    SUM += GAP[i]

In [20]:
SUM

np.float64(0.5113732611989009)

# Q1 Answer

In [21]:
print(round(SUM, 3))

0.511


# Q2

# 2-1

In [22]:
df_2 = WASHER.copy()

In [23]:
df_2.shape

(205, 9)

In [24]:
df_2.columns

Index(['PRD_ID', 'WEIGHT', 'ENERGY_CONS', 'WATER_CONS', 'SPIN_SPEED', 'NOISE',
       'MOTOR_TYPE', 'MATERIAL', 'SCORE'],
      dtype='object')

In [25]:
targets = [ 'WEIGHT', 'ENERGY_CONS', 'WATER_CONS', 'SPIN_SPEED', 'NOISE']

In [26]:
from sklearn.preprocessing import StandardScaler

In [27]:
for target in targets:
    df_2[target + '_S'] = StandardScaler().fit_transform(X=df_2.loc[:, [target]])

In [28]:
df_2.head()

,PRD_ID,WEIGHT,ENERGY_CONS,WATER_CONS,SPIN_SPEED,NOISE,MOTOR_TYPE,MATERIAL,SCORE,WEIGHT_S,ENERGY_CONS_S,WATER_CONS_S,SPIN_SPEED_S,NOISE_S
0,1,251.35,209.10,26.55,1238.97,79.68,A,A,68,-0.306208,0.262800,-0.914288,-1.852161,0.543525
1,2,250.84,208.93,29.32,1240.25,81.78,A,A,75,-0.355352,0.234060,0.626713,-1.823603,1.193455
2,3,254.01,215.46,29.28,1351.34,68.60,A,A,72,-0.049891,1.338015,0.604460,0.654864,-2.885626
3,4,258.55,206.91,25.93,1344.86,77.76,B,B,71,0.387583,-0.107439,-1.259206,0.510292,-0.050696
4,5,258.56,207.93,26.26,1343.22,76.79,C,B,75,0.388546,0.065001,-1.075621,0.473703,-0.350901


# 2-2

In [29]:
tmp = []

for target in targets:
    tmp.append(target + '_S')

In [30]:
targets = tmp

In [31]:
targets

['WEIGHT_S', 'ENERGY_CONS_S', 'WATER_CONS_S', 'SPIN_SPEED_S', 'NOISE_S']

In [32]:
from sklearn.cluster import KMeans

In [33]:
model = KMeans(n_clusters=3, random_state=1234).fit(X=df_2.loc[:, targets])

# 2-3

In [34]:
df_2['LABEL'] = model.predict(X=df_2.loc[:, targets])

In [36]:
summary = df_2.groupby('LABEL')['SCORE'].mean().reset_index()

In [37]:
summary

,LABEL,SCORE
0,0,65.611111
1,1,76.910256
2,2,67.789474


In [38]:
summary['SCORE'].max()

np.float64(76.91025641025641)

# Q2 Answer

In [39]:
print(round(summary['SCORE'].max(), 2))

76.91


# Q3

In [40]:
df_3 = WASHER.copy()

# 3-1

In [41]:
targets = ['MOTOR_TYPE', 'MATERIAL']

In [42]:
df_3 = pd.get_dummies(df_3, columns=targets, drop_first=True)

In [43]:
df_3.head()

,PRD_ID,WEIGHT,ENERGY_CONS,WATER_CONS,SPIN_SPEED,NOISE,SCORE,MOTOR_TYPE_B,MOTOR_TYPE_C,MATERIAL_B
0,1,251.35,209.10,26.55,1238.97,79.68,68,False,False,False
1,2,250.84,208.93,29.32,1240.25,81.78,75,False,False,False
2,3,254.01,215.46,29.28,1351.34,68.60,72,False,False,False
3,4,258.55,206.91,25.93,1344.86,77.76,71,True,False,True
4,5,258.56,207.93,26.26,1343.22,76.79,75,False,True,True


# 3-2

In [44]:
TRAIN26 = df_3[df_3['PRD_ID'] % 5 != 0]
TEST26 = df_3[df_3['PRD_ID'] % 5 == 0]

In [45]:
TRAIN26.shape

(164, 10)

In [46]:
TEST26.shape

(41, 10)

# 3-3

In [47]:
from sklearn.linear_model import LinearRegression

In [48]:
TRAIN26.columns

Index(['PRD_ID', 'WEIGHT', 'ENERGY_CONS', 'WATER_CONS', 'SPIN_SPEED', 'NOISE',
       'SCORE', 'MOTOR_TYPE_B', 'MOTOR_TYPE_C', 'MATERIAL_B'],
      dtype='object')

In [49]:
targets = ['WEIGHT', 'ENERGY_CONS', 'WATER_CONS', 'SPIN_SPEED', 'NOISE', 'MOTOR_TYPE_B', 'MOTOR_TYPE_C', 'MATERIAL_B']

In [50]:
len(targets)

8

In [51]:
model = LinearRegression().fit(y=TRAIN26['SCORE'], X=TRAIN26.loc[:, targets])

# 3-4

In [53]:
pred = model.predict(X=TEST26.loc[:, targets])

In [54]:
from sklearn.metrics import mean_squared_error

In [55]:
mse = mean_squared_error(y_true=TEST26['SCORE'], y_pred=pred)

In [56]:
mse

21.632757503003138

# Q3 Answer

In [57]:
print(round(mse, 3))

21.633
